# B_Bronze - Auto Loader 


In [0]:

from pyspark.sql.functions import col, current_timestamp

CATALOG = "workspace"
BRONZE_SCHEMA = f"{CATALOG}.bronze"

RAW_BASE = "/Volumes/workspace/default/filestore/tables/raw"
CHECKPOINT_BASE = "/Volumes/workspace/default/filestore/checkpoints/bronze"
SCHEMA_BASE = "/Volumes/workspace/default/filestore/schemas/bronze"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")


# ============================================================
# CONFIGURACIÓN DE TABLAS
# ============================================================

TABLES = {

    "calendar": {
        "file_pattern": "calendar*.csv",
        "table": f"{BRONZE_SCHEMA}.calendar",
        "checkpoint": f"{CHECKPOINT_BASE}/calendar/",
        "schema": f"{SCHEMA_BASE}/calendar/",
    },

    "products": {
        "file_pattern": "products*.csv",
        "table": f"{BRONZE_SCHEMA}.products",
        "checkpoint": f"{CHECKPOINT_BASE}/products/",
        "schema": f"{SCHEMA_BASE}/products/",
    },

    "customers": {
        "file_pattern": "customers*.csv",
        "table": f"{BRONZE_SCHEMA}.customers",
        "checkpoint": f"{CHECKPOINT_BASE}/customers/",
        "schema": f"{SCHEMA_BASE}/customers/",
    },

    "stores": {
        "file_pattern": "stores*.csv",
        "table": f"{BRONZE_SCHEMA}.stores",
        "checkpoint": f"{CHECKPOINT_BASE}/stores/",
        "schema": f"{SCHEMA_BASE}/stores/",
    },

    "sales": {
        "file_pattern": "sales*.csv",
        "table": f"{BRONZE_SCHEMA}.sales",
        "checkpoint": f"{CHECKPOINT_BASE}/sales/",
        "schema": f"{SCHEMA_BASE}/sales/",
    },
}


# ============================================================
# FUNCIÓN AUTO LOADER
# ============================================================

def ingest_with_autoloader(cfg):

    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.inferColumnTypes", "true")
        .option(
            "cloudFiles.schemaLocation",
            cfg["schema"]
        )
        .option(
            "pathGlobFilter",
            cfg["file_pattern"]
        )
        .option("header", "true")
        .load(RAW_BASE)
        .withColumn(
            "ingestion_timestamp",
            current_timestamp()
        )
        .withColumn(
            "source_file",
            col("_metadata.file_path")
        )
    )

    query = (
        df.writeStream
        .format("delta")
        .option(
            "checkpointLocation",
            cfg["checkpoint"]
        )
        .option("mergeSchema", "true")
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(cfg["table"])
    )

    query.awaitTermination()


# ============================================================
# EJECUCIÓN
# ============================================================

for name, cfg in TABLES.items():

    print(f"Iniciando Bronze: {name}")

    ingest_with_autoloader(cfg)

    print(
        f"Bronze OK: {cfg['table']}"
    )


print(
    "BRONZE - Auto Loader finalizado correctamente"
)




Iniciando Bronze: calendar
Bronze OK: workspace.bronze.calendar
Iniciando Bronze: products
Bronze OK: workspace.bronze.products
Iniciando Bronze: customers
Bronze OK: workspace.bronze.customers
Iniciando Bronze: stores
Bronze OK: workspace.bronze.stores
Iniciando Bronze: sales
Bronze OK: workspace.bronze.sales
BRONZE - Auto Loader finalizado correctamente


In [0]:
%sql
SELECT 'calendar' AS tabla, COUNT(*) AS total_filas FROM workspace.bronze.calendar
UNION ALL
SELECT 'products' AS tabla, COUNT(*) AS total_filas FROM workspace.bronze.products
UNION ALL
SELECT 'customers' AS tabla, COUNT(*) AS total_filas FROM workspace.bronze.customers
UNION ALL
SELECT 'stores' AS tabla, COUNT(*) AS total_filas FROM workspace.bronze.stores
UNION ALL
SELECT 'sales' AS tabla, COUNT(*) AS total_filas FROM workspace.bronze.sales;


tabla,total_filas
calendar,731
products,200
customers,150015
stores,100
sales,1000000


In [0]:
%sql
SELECT
    source_file,
    COUNT(*) AS cantidad
FROM workspace.bronze.customers
GROUP BY source_file
ORDER BY source_file;

source_file,cantidad
/Volumes/workspace/default/filestore/tables/raw/customers.csv,50000
/Volumes/workspace/default/filestore/tables/raw/customers_2026.csv,50003
/Volumes/workspace/default/filestore/tables/raw/customers_n.csv,2
/Volumes/workspace/default/filestore/tables/raw/customers_new.csv,2
/Volumes/workspace/default/filestore/tables/raw/customers_new2.csv,2
/Volumes/workspace/default/filestore/tables/raw/customers_test.csv,50004
/Volumes/workspace/default/filestore/tables/raw/customers_y.csv,2


In [0]:
%sql
SELECT *
FROM workspace.bronze.customers
WHERE source_file LIKE '%customers_y.csv';

customer_id,age,gender,loyalty_member,join_date,ingestion_timestamp,source_file,_rescued_data
C000015,15,Male,0,2023-06-07,2026-11-26T00:49:24.057Z,/Volumes/workspace/default/filestore/tables/raw/customers_y.csv,null
C000016,16,Male,0,2025-12-22,2026-11-26T00:49:24.057Z,/Volumes/workspace/default/filestore/tables/raw/customers_y.csv,null
